# Part 2: KV-cache eviction and routing

This notebook runs and plots your Part 2 work: LRU, prefix-aware eviction, and cache-aware routing. The [Part 2 handout](../docs/part2_handout.md) is the specification, including the policy API.

Sections 1–3 use one replica, sections 4–5 one and two, and section 6 two. Use the **CS 378 Assignment 4** kernel ([setup](../README.md#setup)).

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt

from agentkv import viz
from agentkv.notebook import check, repo_root

os.chdir(repo_root())          # every path below is relative to the repository root
plt.rcParams["figure.dpi"] = 100

from agentkv import score
from agentkv.harness import CacheHarness
from agentkv.policy import Policy
from agentkv.sandbox.loader import instantiate, load_policy_class


def load_policy(path, **params):
    """Instantiate the policy class defined in a policy file."""
    return instantiate(load_policy_class(Path(path).read_text(), path), params)


def show(h, note=""):
    """Print the cache state: [A*] is active, [B] is cached, and [ ] is empty."""
    active, cached = h.active(), h.cached()
    slots = [f"[{n}*]" for n in active] + [f"[{n}]" for n in cached] + ["[ ]"] * h.empty_slots()
    evicted = f"   evicted so far: {', '.join(h.evicted)}" if h.evicted else ""
    print(f"{' '.join(slots):24} {note}{evicted}")

## 1. Cache states

`CacheHarness` is a small cache with one-token blocks named by letters, for stepping through evictions by hand. `[A*]` is active (held by a running call), `[A]` is cached (kept for reuse), and `[ ]` is empty. The engine fills empty slots before it asks the policy for a victim, and only cached blocks are candidates.

Below, a three-slot cache evicts in FIFO order, which is the order of each block's final release.

In [ ]:
class FIFO(Policy):
    def target_concurrency(self, view, state):
        return 4

    def choose_victim(self, view, candidates, state):
        return candidates[0]          # candidates arrive oldest-released first


h = CacheHarness(FIFO(), capacity=3)
h.allocate("A")
h.allocate("B")
show(h, "allocate A, B")

h.release("A")
h.release("B")
show(h, "release both")

h.allocate("C")
h.release("C")
show(h, "C takes the empty slot")

h.access("A")
h.release("A")
show(h, "hit on A, then A is released again")

h.allocate("D")
show(h, "D needs a slot: FIFO evicts the oldest release, B")

FIFO evicts in release order and LRU in access order. They disagree when a block is used last but released first: `A` is held by one call, `B` by two calls that finish before `A`'s. Which block should go when `D` needs space?

In [ ]:
h = CacheHarness(FIFO(), capacity=3)
h.allocate("A")
h.allocate("B")
h.access("B")
show(h, "A held by one call, B by two")

h.release("B")
h.release("B")
h.release("A")
show(h, "B's calls finish first, then A's")

h.allocate("C")
h.allocate("D")
show(h, "FIFO evicts B, even though B was accessed more recently than A")

## 2. LRU (2.1)

Complete `policies/reference.py` (handout 2.1), run the tests, and replay the same sequence: LRU must evict `A`.

In [ ]:
check("tests/test_part2_lru.py")

In [ ]:
h = CacheHarness(load_policy("policies/reference.py"), capacity=3)
h.allocate("A")
h.allocate("B")
h.access("B")

h.release("B")
h.release("B")
h.release("A")

h.allocate("C")
h.allocate("D")
show(h, "LRU evicts A: B was used more recently")

## 3. LRU on agent traces

`viz.record` replays a trace in the simulator. `dev-W3-0` is document QA: short questions about ~12,000-token documents, with short answers. In the timeline, yellow is waiting for a concurrency slot, orange is waiting in the replica queue, blue is running, and a black tick is a job's arrival. The memory plot shows active (blue), cached (green), and empty (gray) blocks.

In [ ]:
W3 = "data/traces/public_dev/dev-W3-0.json"
rec8 = viz.record(W3, "policies/reference.py", params={"cap": 8})

fig, axes = plt.subplots(2, 1, figsize=(12, 10), gridspec_kw={"height_ratios": [2.4, 1]})
viz.timeline(rec8, axes[0])
viz.memory(rec8, axes[1])
fig.tight_layout()
plt.show()

Predict how lowering the cap changes the hit rate, prefill time, and mean job latency, then run the sweep.

In [ ]:
for cap in (8, 4, 3, 2, 1):
    r = viz.record(W3, "policies/reference.py", params={"cap": cap})
    mean_latency = sum(r.result.flows) / len(r.result.flows)
    hit = sum(c["cached"] for c in r.calls) / sum(c["prompt"] for c in r.calls)
    d = r.result.diag
    print(f"cap {cap}: mean job latency {mean_latency:5.2f} s   prompt tokens from cache {hit:.0%}   "
          f"engine busy {d['busy_time']:4.1f} s, prefilling {d['prefill_time']:4.1f} s")

The same sweep on `dev-D-0`, deep research reports with the parallel branches of Part 1.

In [ ]:
D = "data/traces/public_dev/dev-D-0.json"
for cap in (8, 4, 2):
    r = viz.record(D, "policies/reference.py", params={"cap": cap})
    mean_latency = sum(r.result.flows) / len(r.result.flows)
    hit = sum(c["cached"] for c in r.calls) / sum(c["prompt"] for c in r.calls)
    d = r.result.diag
    print(f"cap {cap}: mean job latency {mean_latency:5.2f} s   prompt tokens from cache {hit:.0%}   "
          f"engine busy {d['busy_time']:4.1f} s, prefilling {d['prefill_time']:4.1f} s")

## 4. Concurrency caps on one and two replicas

LRU at caps 2–16 on the 32 `public_dev` traces, relative to cap 8, first on one replica and then on two (each with the same memory as the single one). Each grid takes several minutes; the coding-agent traces (W2) are slowest. Use both grids for Q1.

![The eight workload families used for evaluation.](../docs/figures/families.svg)

In [ ]:
CAPS = (2, 4, 8, 16)


def cap_grid(replicas):
    # Score LRU at each cap and plot per-family speedups relative to cap 8.
    sweep = {cap: score("policies/reference.py", split="public_dev", replicas=replicas, params={"cap": cap})
             for cap in CAPS}
    bad = [r for r in sweep.values() if not r.valid]
    if bad:
        raise ValueError(f"policies/reference.py is invalid: {bad[0].violations[0]}")
    families = list(sweep[8].by_family)
    grid = [[sweep[c].by_family[f] for c in CAPS] for f in families] + [[sweep[c].speedup for c in CAPS]]
    viz.speedup_grid(grid, families + ["S"], CAPS, xlabel="cap",
                     title=f"LRU speedup over LRU at cap 8, {replicas} replica{'s' if replicas > 1 else ''}")
    plt.show()


cap_grid(replicas=1)

In [ ]:
cap_grid(replicas=2)

## 5. Prefix-aware eviction (2.2)

First, FIFO against LRU on one replica:

In [ ]:
for path in ("policies/fifo.py", "policies/reference.py"):
    result = score(path, split="public_dev", replicas=1)
    print(f"{Path(path).name:13} S = {result.speedup:.3f}  " +
          " ".join(f"{f} {v:.2f}" for f, v in result.by_family.items()))

FIFO wins because a call touches its prompt from root to tail, so LRU evicts a prefix's root first and strands every block after it. Vidur releases a finished call's blocks from tail to root, so FIFO tends to evict tails.

![Evicting a root prevents reuse of its descendants; evicting a tail preserves the remaining prefix.](../docs/figures/evict_tail.svg)

Implement prefix-aware eviction in `policies/prefix_policy.py` (handout 2.2), run the tests, and compare it with LRU on one and two replicas. Use these numbers for Q2.

In [ ]:
check("tests/test_part2_prefix.py")

In [ ]:
rows = {}
for replicas in (1, 2):
    for name, leaf_first in (("LRU", False), ("prefix-aware", True)):
        params = {"leaf_first": leaf_first, "balance_threshold": -1}
        rows[replicas, name] = score("policies/prefix_policy.py", split="public_dev", replicas=replicas,
                                     params=params)
for (replicas, name), r in rows.items():
    print(f"{replicas} replica(s), {name:12} S = {r.speedup:.3f}  " +
          " ".join(f"{f} {v:.2f}" for f, v in r.by_family.items()))

## 6. Cache-aware routing (2.3)

The reference sends each call to the less loaded replica, which balances load but scatters prefixes. Implement `route_call` in `policies/prefix_policy.py` (handout 2.3), then sweep `balance_threshold` with prefix-aware eviction on. Use the grid for Q3.

In [ ]:
THRESHOLDS = (-1, 0, 1, 2, 4, 16)
routing = {t: score("policies/prefix_policy.py", split="public_dev", params={"balance_threshold": t})
           for t in THRESHOLDS}
families = list(routing[-1].by_family)
grid = [[routing[t].by_family[f] for t in THRESHOLDS] for f in families]
grid.append([routing[t].speedup for t in THRESHOLDS])
viz.speedup_grid(grid, families + ["S"], [str(t) for t in THRESHOLDS],
                 xlabel="balance_threshold (-1: least loaded)",
                 title="Prefix-aware eviction with cache-aware routing, 2 replicas")
plt.show()

A customer-service trace, where every call shares one long policy prompt, at `balance_threshold` 1 and 16.

In [ ]:
W4 = "data/traces/public_dev/dev-W4-0.json"
fig, axes = plt.subplots(2, 1, figsize=(12, 11), layout="constrained")
for ax, threshold in zip(axes, (1, 16)):
    rec = viz.record(W4, "policies/prefix_policy.py", params={"balance_threshold": threshold}, replicas=2)
    mean = sum(rec.result.flows) / len(rec.result.flows)
    viz.timeline(rec, ax, title=f"balance_threshold = {threshold}: mean job latency {mean:.2f} s")
plt.show()

## 7. Optional: job-first dispatch

`dispatch_order` chooses which ready calls go next. With two 3-call jobs arriving together, 1 s per call, and `cap = 1`, interleaving them (`A1, B1, A2, B2, A3, B3`) finishes the jobs at 5 s and 6 s; running the older job first finishes them at 3 s and 6 s. To try it, copy `policies/prefix_policy.py` to `candidates/job_first.py`, add this hook, and score both:

```python
    def dispatch_order(self, view, ready, state):
        return sorted(ready, key=lambda c: view.jobs[view.ready[c].job].arrival)
```

## Questions

Answer Q1–Q3 from the [handout](../docs/part2_handout.md#24-experiments-and-questions) in `assignment4_report.pdf`.